# **ЛАБОРАТОРНА РОБОТА 6**
## Робота з відкритим програмним кодом

In [11]:
import sys
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from segment_anything import (sam_model_registry,
                              SamAutomaticMaskGenerator, SamPredictor)

import urllib.request
import os

In [12]:
url = "https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg"
urllib.request.urlretrieve(url, "image.jpg")

('image.jpg', <http.client.HTTPMessage at 0x2489774b8d0>)

In [13]:
CHECKPOINT = "sam_vit_b_01ec64.pth"
MODEL_TYPE = "vit_b"
IMAGE_PATH = "image.jpg"

device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT)
sam.to(device=device)

bgr = cv2.imread(IMAGE_PATH)
if bgr is None:
    raise FileNotFoundError(IMAGE_PATH)
image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

In [14]:
def overlay(img, masks, alpha=0.5):
    
    out = img.copy().astype(np.float32)
    rng = np.random.default_rng(0)
    for m in sorted(masks, key=lambda x: x["area"], reverse=True):
        color = rng.integers(0, 255, 3)
        seg = m["segmentation"]
        out[seg] = (1 - alpha) * out[seg] + alpha * color
    return out.astype(np.uint8)

generator = SamAutomaticMaskGenerator(sam)
masks = generator.generate(image)

print("Automatic mode: found", len(masks), "masks")

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(overlay(image, masks))
plt.title("Automatic masks: %d" % len(masks))
plt.axis("off")
plt.tight_layout()
plt.savefig("result_auto.png", dpi=150)
plt.close()

Automatic mode: found 44 masks


In [15]:
predictor = SamPredictor(sam)
predictor.set_image(image)

h, w = image.shape[:2]
point = np.array([[w // 2, h // 2]])  
label = np.array([1])                 

mask_list, scores, _ = predictor.predict(
    point_coords=point,
    point_labels=label,
    multimask_output=True,            
)

plt.figure(figsize=(15, 5))
for i, (m, s) in enumerate(zip(mask_list, scores)):
    plt.subplot(1, 3, i + 1)
    plt.imshow(image)
    plt.imshow(m, alpha=0.5, cmap="jet")
    plt.scatter(point[:, 0], point[:, 1], c="lime", s=100, marker="*")
    plt.title("Mask %d, score %.3f" % (i + 1, s))
    plt.axis("off")
plt.tight_layout()
plt.savefig("result_prompt.png", dpi=150)
plt.close()
print("Saved result_auto.png and result_prompt.png")

Saved result_auto.png and result_prompt.png
